In [2]:
import json
import os
import pickle
import time
import numpy as np
import pandas as pd
from datetime import datetime
import random
import torch

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

sns.set_style("whitegrid")
print("Libraries imported successfully.")

DATASET = "SST-2"
MODEL = "LogReg"
C_VALUE = 10.0
N_ITERATIONS = 150
TEST_LABELS = False
SEED = 42

MAX_FEATURES = 50000
NGRAM_RANGE = (1, 2)
SUBLINEAR_TF = True
STRIP_ACCENTS = "unicode"

TEXT_COLUMN = "sentence_preprocessed"
LABEL_COLUMN = "label"
ROOT_DIR = os.getcwd()

print(f"Dataset:      {DATASET}")
print(f"Max features: {MAX_FEATURES}")
print(f"Ngram range:  {NGRAM_RANGE}")
print(f"Sublinear TF: {SUBLINEAR_TF}")

print(f"Strip accents:{STRIP_ACCENTS}")
print(f"Test labels:  {TEST_LABELS}")
print(f"Seed:         {SEED}")
print(f"Text column:  {TEXT_COLUMN}")
print(f"Label column: {LABEL_COLUMN}")
print(f"Root dir:     {ROOT_DIR}")

Libraries imported successfully.
Dataset:      SST-2
Max features: 50000
Ngram range:  (1, 2)
Sublinear TF: True
Strip accents:unicode
Test labels:  False
Seed:         42
Text column:  sentence_preprocessed
Label column: label
Root dir:     c:\Research\NLP_FAC


In [3]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))

Loading data from: c:\Research\NLP_FAC\SST-2

  train.json: 67349 samples
  validation.json: 872 samples
  test.json: 1821 samples

First 3 train examples:


,sentence_original,sentence_preprocessed,label
0,hide new secretions from the parental units,hide new secretions from the parental units,0
1,"contains no wit , only labored gags","contains no wit , only labored gags",0
2,that loves its characters and communicates som...,that loves its characters and communicates som...,1


In [4]:
X_train_text = df_train[TEXT_COLUMN].fillna("").values
y_train = df_train[LABEL_COLUMN].values

X_val_text = df_validation[TEXT_COLUMN].fillna("").values
y_val = df_validation[LABEL_COLUMN].values

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=SUBLINEAR_TF,
    strip_accents=STRIP_ACCENTS
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

print(f"TF-IDF vocabulary: {len(tfidf.vocabulary_)} features")
print(f"X_train shape: {X_train_tfidf.shape}")
print(f"X_val shape:   {X_val_tfidf.shape}")
print(f"Unique classes: {np.unique(y_train)}")

TF-IDF vocabulary: 50000 features
X_train shape: (67349, 50000)
X_val shape:   (872, 50000)
Unique classes: [0 1]


In [5]:
def build_model(model_name, C, max_iter):
    if model_name == "LogReg":
        model = LogisticRegression(
            C=C, max_iter=max_iter, solver="lbfgs",
            random_state=SEED
        )
        params = {"type": "LogisticRegression", "C": C,
                  "max_iter": max_iter, "solver": "lbfgs"}
        loss_type = "log_loss"
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model, params, loss_type

def compute_loss(model, X, y, loss_type):
    if loss_type == "log_loss":
        y_proba = model.predict_proba(X)
        return log_loss(y, y_proba)
    else:
        raise ValueError(f"Unsupported loss type: {loss_type}")


print(f"Building model: {MODEL}")
model, model_params, loss_type = build_model(MODEL, C_VALUE, N_ITERATIONS)
print(f"Model params: {model_params}")

Building model: LogReg
Model params: {'type': 'LogisticRegression', 'C': 10.0, 'max_iter': 150, 'solver': 'lbfgs'}


In [6]:
X_train_final = X_train_tfidf
X_val_final = X_val_tfidf

X_test_text = df_test[TEXT_COLUMN].fillna("").values
X_test_final = tfidf.transform(X_test_text)

y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].values

train_losses, val_losses, test_losses = [], [], []
train_accs, val_accs, test_accs = [], [], []

print(f"\nTraining {MODEL} -- max_iter from 1 to {N_ITERATIONS}...\n")
start_time = time.time()

for it in range(1, N_ITERATIONS + 1):
    model, model_params, loss_type = build_model(MODEL, C_VALUE, max_iter=it)
    model.fit(X_train_final, y_train)

    y_tr_pred = model.predict(X_train_final)
    y_vl_pred = model.predict(X_val_final)

    train_accs.append(accuracy_score(y_train, y_tr_pred))
    val_accs.append(accuracy_score(y_val, y_vl_pred))
    train_losses.append(compute_loss(model, X_train_final, y_train, loss_type))
    val_losses.append(compute_loss(model, X_val_final, y_val, loss_type))

    if TEST_LABELS:
        y_ts_pred = model.predict(X_test_final)
        test_accs.append(accuracy_score(y_test, y_ts_pred))
        test_losses.append(compute_loss(model, X_test_final, y_test, loss_type))

    if it % 10 == 0 or it == 1:
        test_info = f" | Test Acc: {test_accs[-1]:.4f}" if TEST_LABELS else ""
        print(f"  Iter {it:3d}/{N_ITERATIONS} | "
              f"Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | "
              f"Train Acc: {train_accs[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}{test_info}")

train_time = time.time() - start_time
entries_per_second = (len(y_train) * N_ITERATIONS) / train_time

print(f"\nTraining complete in {train_time:.2f} seconds")
print(f"  Entries/sec: {entries_per_second:.0f}")
print(f"  Final Train Acc: {train_accs[-1]:.4f}")
print(f"  Final Val Acc:   {val_accs[-1]:.4f}")
if TEST_LABELS:
    print(f"  Final Test Acc:  {test_accs[-1]:.4f}")


Training LogReg -- max_iter from 1 to 150...

  Iter   1/150 | Train Loss: 0.6862 | Val Loss: 0.6975 | Train Acc: 0.5578 | Val Acc: 0.5092


C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/prepr

  Iter  10/150 | Train Loss: 0.4502 | Val Loss: 0.5503 | Train Acc: 0.8081 | Val Acc: 0.7546


C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 11 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=11).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 12 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=12).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

  Iter  20/150 | Train Loss: 0.3005 | Val Loss: 0.4935 | Train Acc: 0.8882 | Val Acc: 0.8028


...

  Iter  90/150 | Train Loss: 0.1117 | Val Loss: 0.3972 | Train Acc: 0.9749 | Val Acc: 0.8257


C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 91 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=91).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 92 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=92).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

  Iter 100/150 | Train Loss: 0.1108 | Val Loss: 0.3947 | Train Acc: 0.9752 | Val Acc: 0.8280


C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 101 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=101).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 102 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=102).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modul

  Iter 110/150 | Train Loss: 0.1118 | Val Loss: 0.3935 | Train Acc: 0.9760 | Val Acc: 0.8291
  Iter 120/150 | Train Loss: 0.1118 | Val Loss: 0.3935 | Train Acc: 0.9760 | Val Acc: 0.8291
  Iter 130/150 | Train Loss: 0.1118 | Val Loss: 0.3935 | Train Acc: 0.9760 | Val Acc: 0.8291
  Iter 140/150 | Train Loss: 0.1118 | Val Loss: 0.3935 | Train Acc: 0.9760 | Val Acc: 0.8291
  Iter 150/150 | Train Loss: 0.1118 | Val Loss: 0.3935 | Train Acc: 0.9760 | Val Acc: 0.8291

Training complete in 100.44 seconds
  Entries/sec: 100578
  Final Train Acc: 0.9760
  Final Val Acc:   0.8291


In [7]:
y_train_pred_final = model.predict(X_train_final)
y_val_pred = model.predict(X_val_final)
y_test_pred = model.predict(X_test_final)

train_accuracy = accuracy_score(y_train, y_train_pred_final)
train_loss_final = train_losses[-1]

val_accuracy = accuracy_score(y_val, y_val_pred)
val_precision_macro = precision_score(y_val, y_val_pred, average="macro", zero_division=0)
val_recall_macro = recall_score(y_val, y_val_pred, average="macro", zero_division=0)
val_f1_macro = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
val_precision_per_class = precision_score(y_val, y_val_pred, average=None, zero_division=0).tolist()
val_recall_per_class = recall_score(y_val, y_val_pred, average=None, zero_division=0).tolist()
val_f1_per_class = f1_score(y_val, y_val_pred, average=None, zero_division=0).tolist()
val_conf_matrix = confusion_matrix(y_val, y_val_pred).tolist()
val_loss_final = val_losses[-1]

class_labels = sorted(np.unique(y_val).tolist())

if TEST_LABELS:
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision_macro = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_recall_macro = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_precision_per_class = precision_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_recall_per_class = recall_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_f1_per_class = f1_score(y_test, y_test_pred, average=None, zero_division=0).tolist()
    test_conf_matrix = confusion_matrix(y_test, y_test_pred).tolist()
    test_loss_final = test_losses[-1]

print("=" * 60)
print(f"  RESULTS {MODEL} on {DATASET}")
print("=" * 60)
print(f"  Train Accuracy:      {train_accuracy:.4f}   | Loss: {train_loss_final:.4f}")
print(f"  Val Accuracy:        {val_accuracy:.4f}   | Loss: {val_loss_final:.4f}")
if TEST_LABELS:
    print(f"  Test Accuracy:       {test_accuracy:.4f}   | Loss: {test_loss_final:.4f}")
    print(f"  Test Precision:      {test_precision_macro:.4f}")
    print(f"  Test Recall:         {test_recall_macro:.4f}")
    print(f"  Test F1 (macro):     {test_f1_macro:.4f}")
    print(f"\n  Test Confusion Matrix: {test_conf_matrix}")
    print(f"\n  Classification Report (TEST):")
    print(classification_report(y_test, y_test_pred, zero_division=0))
else:
    unique, counts = np.unique(y_test_pred, return_counts=True)
    print(f"  Test prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")
    print(f"\n  Classification Report (VALIDATION):")
    print(classification_report(y_val, y_val_pred, zero_division=0))

  RESULTS LogReg on SST-2
  Train Accuracy:      0.9760   | Loss: 0.1118
  Val Accuracy:        0.8291   | Loss: 0.3935
  Test prediction distribution: {0: 845, 1: 976}

  Classification Report (VALIDATION):
              precision    recall  f1-score   support

           0       0.86      0.78      0.82       428
           1       0.81      0.87      0.84       444

    accuracy                           0.83       872
   macro avg       0.83      0.83      0.83       872
weighted avg       0.83      0.83      0.83       872



In [9]:
if TEST_LABELS:
    cm_array = np.array(test_conf_matrix)
    cm_labels = sorted(np.unique(y_test).tolist())
else:
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val).tolist())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
ax_cm.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
fig_cm.tight_layout()
plt.show()

print(f"Classes: {cm_labels}")
print(f"Matrix:\n{cm_array}")

Classes: [0, 1]
Matrix:
[[335  93]
 [ 56 388]]


C:\Users\User\AppData\Local\Temp\ipykernel_18252\3610909509.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
iters_range = list(range(1, N_ITERATIONS + 1))

fig_acc, ax_acc = plt.subplots(figsize=(9, 5))
ax_acc.plot(iters_range, train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc.plot(iters_range, val_accs, label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc.set_xlabel("Iteration (max_iter)")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_acc.legend(loc="lower right")
ax_acc.grid(True, alpha=0.3)
fig_acc.tight_layout()
plt.show()

print(f"Last iteration -- Train: {train_accs[-1]:.4f}, Val: {val_accs[-1]:.4f}")

Last iteration -- Train: 0.9760, Val: 0.8291


C:\Users\User\AppData\Local\Temp\ipykernel_18252\1311820378.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig_loss, ax_loss = plt.subplots(figsize=(9, 5))
ax_loss.plot(iters_range, train_losses, label="Train Loss", linewidth=2, color="#1f77b4")
ax_loss.plot(iters_range, val_losses, label="Validation Loss", linewidth=2, color="#ff7f0e")
ax_loss.set_xlabel("Iteration (max_iter)")
ax_loss.set_ylabel(f"Loss ({loss_type})")
ax_loss.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_loss.legend(loc="upper right")
ax_loss.grid(True, alpha=0.3)
fig_loss.tight_layout()
plt.show()

print(f"Last iteration -- Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

Last iteration -- Train Loss: 0.1118, Val Loss: 0.3935


C:\Users\User\AppData\Local\Temp\ipykernel_18252\1771664987.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
counts = [np.sum(y_val_pred == 0), np.sum(y_val_pred == 1)]
labels_bar = ["Negative (0)", "Positive (1)"]
colors = ["#e74c3c", "#2ecc71"]

fig_dist, ax_dist = plt.subplots(figsize=(6, 5))
bars = ax_dist.bar(labels_bar, counts, color=colors, edgecolor="black", width=0.5)
for bar, count in zip(bars, counts):
    ax_dist.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 str(count), ha="center", va="bottom", fontweight="bold", fontsize=12)
ax_dist.set_xlabel("Predicted Sentiment")
ax_dist.set_ylabel("Number of Samples")
ax_dist.set_title(f"{MODEL} - {DATASET}", fontweight="bold")
fig_dist.tight_layout()
plt.show()

print(f"Negative: {counts[0]}, Positive: {counts[1]}")

Negative: 391, Positive: 481


C:\Users\User\AppData\Local\Temp\ipykernel_18252\3325034338.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
TOP_N = 20

feature_names = tfidf.get_feature_names_out()
coefs = model.coef_[0]

top_pos_idx = np.argsort(coefs)[-TOP_N:]
top_neg_idx = np.argsort(coefs)[:TOP_N]

fig_words, (ax_neg, ax_pos) = plt.subplots(1, 2, figsize=(14, 6))

ax_neg.barh(feature_names[top_neg_idx], coefs[top_neg_idx], color="#e74c3c")
ax_neg.set_xlabel("Coefficient Value")
ax_neg.set_title(f"Top {TOP_N} Negative Words", fontweight="bold")
ax_neg.invert_yaxis()

ax_pos.barh(feature_names[top_pos_idx], coefs[top_pos_idx], color="#2ecc71")
ax_pos.set_xlabel("Coefficient Value")
ax_pos.set_title(f"Top {TOP_N} Positive Words", fontweight="bold")

fig_words.suptitle(f"{MODEL} - {DATASET}", fontweight="bold", fontsize=14)
fig_words.tight_layout()
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_18252\1534955058.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
def save_results(model, model_params, loss_type, tfidf, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, c_value, n_iterations, test_labels,
                 text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir):

    model_bytes = pickle.dumps(model)
    model_size_bytes = len(model_bytes)
    model_size_mb = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name = f"C{c_value}_iters{n_iterations}_loss-{loss_type}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    tfidf_params = tfidf.get_params()
    metrics = {
        "model": model_name,
        "dataset": dataset,
        "run_name": run_name,
        "timestamp": timestamp_str,
        "model_params": model_params,
        "seed": SEED,
        "model_size": {
            "bytes": model_size_bytes,
            "MB": round(model_size_mb, 4)
        },
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second": round(entries_per_second, 2),
            "num_iterations": n_iterations,
            "train_samples": len(df_train),
        },
        "vectorization": {
            "method": "TF-IDF",
            "max_features": tfidf_params.get("max_features"),
            "ngram_range": list(tfidf_params.get("ngram_range", ())),
            "sublinear_tf": tfidf_params.get("sublinear_tf"),
            "vocabulary_size": len(tfidf.vocabulary_)
        },
        "data": {
            "train_samples": len(df_train),
            "validation_samples": len(df_validation),
            "test_samples": len(df_test) if df_test is not None else 0,
            "text_column": text_column,
            "label_column": label_column,
            "classes": class_labels
        },
        "train_results": train_results,
        "validation_results": val_results,
    }

    if test_labels:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name


figures = {
    "confusion_matrix.png": fig_cm,
    "accuracy_curves.png": fig_acc,
    "loss_curves.png": fig_loss,
    "prediction_distribution.png": fig_dist,
    "top_words.png": fig_words,
}

train_res = {
    "accuracy": round(train_accuracy, 4),
    "loss": round(train_loss_final, 4),
}

val_res = {
    "accuracy": round(val_accuracy, 4),
    "loss": round(val_loss_final, 4),
    "precision_macro": round(val_precision_macro, 4),
    "recall_macro": round(val_recall_macro, 4),
    "f1_macro": round(val_f1_macro, 4),
    "precision_per_class": [round(p, 4) for p in val_precision_per_class],
    "recall_per_class": [round(r, 4) for r in val_recall_per_class],
    "f1_per_class": [round(f, 4) for f in val_f1_per_class],
    "confusion_matrix": val_conf_matrix,
}

test_res = None
if TEST_LABELS:
    test_res = {
        "accuracy": round(test_accuracy, 4),
        "loss": round(test_loss_final, 4),
        "precision_macro": round(test_precision_macro, 4),
        "recall_macro": round(test_recall_macro, 4),
        "f1_macro": round(test_f1_macro, 4),
        "precision_per_class": [round(p, 4) for p in test_precision_per_class],
        "recall_per_class": [round(r, 4) for r in test_recall_per_class],
        "f1_per_class": [round(f, 4) for f in test_f1_per_class],
        "confusion_matrix": test_conf_matrix,
    }

output_dir, metrics_path, run_name = save_results(
    model=model, model_params=model_params, loss_type=loss_type, tfidf=tfidf,
    figures=figures,
    train_results=train_res, val_results=val_res, test_results=test_res,
    train_time=train_time, entries_per_second=entries_per_second,
    dataset=DATASET, model_name=MODEL, c_value=C_VALUE, n_iterations=N_ITERATIONS,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,
)

Charts saved: ['confusion_matrix.png', 'accuracy_curves.png', 'loss_curves.png', 'prediction_distribution.png', 'top_words.png']
metrics.json saved in: c:\Research\NLP_FAC\LogReg\SST-2\C10.0_iters150_loss-log_loss_26_03_2026_09_46_21


In [15]:
with open(metrics_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{'='*60}")
print(f"  RUN SUMMARY: {run_name}")
print(f"{'='*60}")
print(f"  Model:              {saved['model']}")
print(f"  Dataset:            {saved['dataset']}")
print(f"  Training time:      {saved['training']['training_time_seconds']}s")
print(f"  Entries/sec:        {saved['training']['entries_per_second']}")
print(f"  Model size:         {saved['model_size']['MB']} MB")
print(f"  Train Accuracy:     {saved['train_results']['accuracy']}")
print(f"  Train Loss:         {saved['train_results']['loss']}")
print(f"  Val Accuracy:       {saved['validation_results']['accuracy']}")
print(f"  Val Loss:           {saved['validation_results']['loss']}")
print(f"  Val F1 (macro):     {saved['validation_results']['f1_macro']}")
if "test_results" in saved:
    print(f"  Test Accuracy:      {saved['test_results']['accuracy']}")
    print(f"  Test Loss:          {saved['test_results']['loss']}")
    print(f"  Test F1 (macro):    {saved['test_results']['f1_macro']}")
else:
    print(f"  Test:               N/A (TEST_LABELS=False)")
print(f"{'='*60}")
print(f"\n  Files saved in: {output_dir}")
for f_name in os.listdir(output_dir):
    f_size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"    {f_name} ({f_size:,} bytes)")

  RUN SUMMARY: C10.0_iters150_loss-log_loss_26_03_2026_09_46_21
  Model:              LogReg
  Dataset:            SST-2
  Training time:      100.4434s
  Entries/sec:        100577.58
  Model size:         0.3821 MB
  Train Accuracy:     0.976
  Train Loss:         0.1118
  Val Accuracy:       0.8291
  Val Loss:           0.3935
  Val F1 (macro):     0.8285
  Test:               N/A (TEST_LABELS=False)

  Files saved in: c:\Research\NLP_FAC\LogReg\SST-2\C10.0_iters150_loss-log_loss_26_03_2026_09_46_21
    accuracy_curves.png (50,233 bytes)
    confusion_matrix.png (24,544 bytes)
    loss_curves.png (55,256 bytes)
    metrics.json (1,478 bytes)
    prediction_distribution.png (27,654 bytes)
    top_words.png (86,262 bytes)


In [16]:
print(json.dumps(saved, indent=2, ensure_ascii=False))

{
  "model": "LogReg",
  "dataset": "SST-2",
  "run_name": "C10.0_iters150_loss-log_loss_26_03_2026_09_46_21",
  "timestamp": "26_03_2026_09_46_21",
  "model_params": {
    "type": "LogisticRegression",
    "C": 10.0,
    "max_iter": 150,
    "solver": "lbfgs"
  },
  "seed": 42,
  "model_size": {
    "bytes": 400678,
    "MB": 0.3821
  },
  "training": {
    "training_time_seconds": 100.4434,
    "entries_per_second": 100577.58,
    "num_iterations": 150,
    "train_samples": 67349
  },
  "vectorization": {
    "method": "TF-IDF",
    "max_features": 50000,
    "ngram_range": [
      1,
      2
    ],
    "sublinear_tf": true,
    "vocabulary_size": 50000
  },
  "data": {
    "train_samples": 67349,
    "validation_samples": 872,
    "test_samples": 1821,
    "text_column": "sentence_preprocessed",
    "label_column": "label",
    "classes": [
      0,
      1
    ]
  },
  "train_results": {
    "accuracy": 0.976,
    "loss": 0.1118
  },
  "validation_results": {
    "accuracy": 0.8291

In [17]:

# Find misclassified validations
misclassified_idx = np.where(y_val != y_val_pred)[0]

output_file = os.path.join(ROOT_DIR, "misclassified_reviews.txt")
with open(output_file, "w", encoding="utf-8") as f:
        f.write("Misclassified Reviews\n")
        f.write("=====================\n\n")
        for idx in misclassified_idx:
                text = df_validation.iloc[idx]['sentence_original']
                actual = y_val[idx]
                predicted = y_val_pred[idx]
        
                actual_str = "Good (1)" if actual == 1 else "Bad (0)"
                predicted_str = "Good (1)" if predicted == 1 else "Bad (0)"
        
                f.write(f"Review: {text}\n")
                f.write(f"Actual: {actual_str}\n")
                f.write(f"Predicted: {predicted_str}\n")
                f.write("-" * 50 + "\n")
        
print(f"Saved {len(misclassified_idx)} misclassified reviews to {output_file}")


Saved 149 misclassified reviews to c:\Research\NLP_FAC\misclassified_reviews.txt
